# Binarization with scBoolSeq

1) Set the GO per macrostate for the evaluation of the HVG and binarization results 
2) Binarize the matrix, the workflow is based on the raw matrix, and the macrostates are binarized separately
3) Evaluate the binarization result 

In [22]:
import sys
#!pip install scboolseq
import scanpy as sc
import numpy as np
import pandas as pd
from scboolseq import scBoolSeq
import gc
import warnings

In [33]:
adata = sc.read("/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Matrix/cll_myel_macro_stream.h5ad")

In [34]:
adata.X.max()

np.float64(7288.0)

In [36]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [37]:
adata.X.max()

np.float64(8.008315932903622)

1) Evaluate the GO per macrostate (GO before HVG and binarization)

In [40]:
adata.obs["macrostates"].head()

D1_AAACCCAAGTCCCGAC-1    None
D1_AAACCCACATAGCACT-1    None
D1_AAACCCACATGGCCCA-1      I1
D1_AAACCCAGTCTTTCTA-1    None
D1_AAACCCAGTTCTAAGC-1    None
Name: macrostates, dtype: category
Categories (6, object): ['I1', 'I2', 'None', 'T1', 'T2', 'T3']

2) Binarization with scBoolSeq

Binarization macrostates separately

In [39]:
adata.obs['macrostates'] = adata.obs['macrostates'].replace({
    'other': "None",
    'initial_1': 'I1',
    'initial_2': 'I2',
    'terminal_1': 'T1',
    'terminal_2': 'T2',
    'terminal_3': 'T3'
})

In [12]:
mapping = {0: "I1",1: "I2",2: "T1",3: "T2",4: "T3",5: "None"}

adata.obs["macrostates"] = adata.obs["macrostates"].map(mapping)
print(adata.obs["macrostates"].unique())

['I2' 'None' 'I1' 'T1' 'T3' 'T2']


In [41]:
warnings.filterwarnings("ignore")
macrostates = ['I1', 'I2', 'T1', 'T2', "T3"]

binarized_states = {}
all_hvg = set()
adata_ct_dict = {}

for ct in macrostates:
    print(f"\n{'='*50}")
    print(f"Processing {ct}...")
    
    adata_ct = adata[adata.obs['macrostates'] == ct].copy()
    n_cells = adata_ct.n_obs
    print(f"  {n_cells} cells")

    # STEP 1 : HVG adapted to the cell type size
    n_top = min(2000, adata_ct.n_vars - 1)
    sc.pp.highly_variable_genes(adata_ct, n_top_genes=n_top)

    adata_ct_hvg = adata_ct[:, adata_ct.var['highly_variable']].copy()
    hvg_genes = adata_ct.var[adata_ct.var['highly_variable']].index
    all_hvg.update(hvg_genes)
    print(f"  {adata_ct_hvg.n_vars} HVGs selected")

    # STEP 2 : Building the DataFrame for scBoolSeq
    X_full = adata_ct_hvg.X
    if not isinstance(X_full, np.ndarray):
        X_full = X_full.toarray()

    expr_df_full = pd.DataFrame(
        X_full,
        index=adata_ct_hvg.obs_names,
        columns=adata_ct_hvg.var_names
    )

    # Eliminate the 0 genes
    expr_df_full = expr_df_full.loc[:, (expr_df_full != 0).any(axis=0)]
    print(f" {expr_df_full.shape[1]} genes after removing all-zero")

    # STEP 3 : Fit scBoolSeq on all the cell of the macrostate
    scbool = scBoolSeq(
        zeroinf_binarizer="quantile",# use quantile thresholds for zero-inflated genes
        margin_quantile=0.1, # 10th/90th percentile instead of 5th/95th
        dor_threshold=0.85, # classify as zero-inflated if >85% zeros
        alpha=0.0, # no IQR expansion (keep it simple)
    )
    scbool.fit(expr_df_full)
    print(f"  scBoolSeq fitted")

    # STEP 4 : Binarize per macrostate
    binarized = scbool.binarize(expr_df_full)

    # STEP 5 : Majority vote sur toutes les cellules
    def majority_vote(col):
        valid = col.dropna()
        if len(valid) == 0:
            return np.nan
        frac_ones = (valid == 1).mean()
        if frac_ones >= 0.5:
            return 1
        elif (valid == 0).mean() >= 0.5:
            return 0
        else:
            return np.nan

    aggregated = binarized.apply(majority_vote, axis=0)
    binarized_states[ct] = aggregated

    n_ones  = (aggregated == 1).sum()
    n_zeros = (aggregated == 0).sum()
    n_nan   = aggregated.isna().sum()
    print(f"  {ct}: {n_cells} cells → {n_ones} genes=1, {n_zeros} genes=0, {n_nan} genes=NaN")

    del adata_ct, expr_df_full, scbool
    gc.collect()

print("\nDone!")


Processing I1...
  397 cells
  2000 HVGs selected
 2000 genes after removing all-zero
  scBoolSeq fitted
  I1: 397 cells → 1030 genes=1, 53 genes=0, 917 genes=NaN

Processing I2...
  225 cells
  2000 HVGs selected
 2000 genes after removing all-zero
  scBoolSeq fitted
  I2: 225 cells → 916 genes=1, 149 genes=0, 935 genes=NaN

Processing T1...
  114 cells
  2000 HVGs selected
 2000 genes after removing all-zero
  scBoolSeq fitted
  T1: 114 cells → 471 genes=1, 34 genes=0, 1495 genes=NaN

Processing T2...
  217 cells
  2000 HVGs selected
 2000 genes after removing all-zero
  scBoolSeq fitted
  T2: 217 cells → 354 genes=1, 200 genes=0, 1446 genes=NaN

Processing T3...
  443 cells
  2000 HVGs selected
 2000 genes after removing all-zero
  scBoolSeq fitted
  T3: 443 cells → 606 genes=1, 202 genes=0, 1192 genes=NaN

Done!


In [42]:
pd.DataFrame.from_dict(binarized_states, orient="index").fillna('')

,ISG15,TNFRSF4,SCNN1D,PEX10,PER3,ENO1,KIF1B,LRRC38,PDPN,KAZN,...,TCEAL3,NUP62CL,DCX,IL13RA1,ELF4,LDOC1,MAMLD1,MTMR1,PNMA5,ZFP92
I1,1.0,,,,1.0,1.0,1.0,,,,...,,,,,,,,,,
I2,1.0,,,,,1.0,,,,,...,,,,,,,,,,
T1,,,,,,,,,,,...,,,,,,,,,,
T3,1.0,1.0,,,1.0,,,,,,...,1.0,,,,,,,1.0,,
T2,,,,,,,,,,,...,,,,,,,,,,


In [43]:
print("Building global HVG matrix...")
all_hvg = list(all_hvg)
adata_hvg = adata[:, all_hvg].copy()

X = adata_hvg.X
if not isinstance(X, np.ndarray):
    X = X.toarray()

labels = adata_hvg.obs['macrostates'].values
print("Compute intra variance...")
intra_var = []
for c in np.unique(labels):
    idx = np.where(labels == c)[0]
    if len(idx) > 1:
        intra_var.append(np.mean(np.var(X[idx, :], axis=0)))

intra_var_mean = np.mean(intra_var)
print("Compute inter variance...")
means_per_cluster = np.array([
    X[np.where(labels == c)[0], :].mean(axis=0)
    for c in np.unique(labels)
])

inter_var = np.mean(np.var(means_per_cluster, axis=0))
ratio = inter_var / intra_var_mean if intra_var_mean > 0 else np.nan

print(f"Inter/Intra macrostate ratio: {ratio:.4f}")

Building global HVG matrix...
Compute intra variance...
Compute inter variance...
Inter/Intra macrostate ratio: 0.1149
